# April experiment — NVDA, INTC & IBM, fully separated

Runs the experiment end to end from raw data, reproducing the colleague's two
current drivers (his 2026-02 `LearningKraus.py` and
`LearningKraus_multivariate.py`, which he ran for AAPL) per symbol:

1. **Configs** — every April-2025 trading day on disk, **one config per
   (symbol, training group)** — the two drivers use different hyperparameters
   and one config holds one training block. `instrument_filter: true` so NVDA,
   INTC, and IBM are **entirely separated** (own filtered event stream, own
   feature cache, own distributions, own models — they share nothing but the
   read-only raw files). NVDA and INTC read from `data/NVDA_INTC`
   (interleaved); IBM reads from its own `data/IBM` (`data.asset_paths` in
   `configs/default.yaml` maps each symbol to its directory).
2. **Distributions** — featurize → encode → `SEQ_DISTR_*`/`CLS_DISTR_*` per
   symbol (once per symbol; the group configs share every distribution
   setting). Includes the joint multivariate `SEQ_DISTR_*` file.
3. **Training** — 3 symbols × (3 bivariate + 1 multivariate) = **12 models**,
   trained by the pipeline's registered trainer (`pipeline.models` — the same
   code path as `python -m pipeline train`) and fanned one per GPU. Per model
   you get the **2 result files + 4 charts** (12×2 = 24 files, 12×4 = 48
   images), shown inline below as each model completes — forward each bundle
   as soon as it appears.

Every parameter comes from the generated config, printed in full by the config cell
before anything runs — that printout is the authoritative record. Training is unseeded
(like his drivers) unless you set `SEED`.

*Long runs over SSH: JupyterLab keeps the kernel alive if the browser disconnects; from a
plain terminal you can run the same thing detached with `scripts/run_april.py` under
`nohup`/`tmux`.*

In [ ]:
# ---- parameters, set for the LINUX COMPUTE ENVIRONMENT ----
# (training groups = the colleague's two 2026-02 drivers, verbatim;
#  distribution/ensemble scope = his email spec + the multivariate entry)
SYMBOLS = ["NVDA", "INTC", "IBM"]                 # per boss: all three, separated
PREDICTED = "log_mid"    # features_list[0] in ALL colleague files: every SEQ/CLS/ENS
                         # output pairs log_mid with predictor(s) — log_mid is
                         # the thing being predicted, never a "predictor" itself

# TRAINING: one group per colleague driver; each becomes its own generated
# config (the groups differ in optimizer/batch/qubits/max_seq_len, and one
# config holds one training block). These dicts only feed make_config —
# stage 3 reads everything back from the generated YAMLs printed below.
#   group 0 — his LearningKraus.py bivariate script body: its opt_name
#     reassignment chain lands on sgd, batch 8*512, 3 qubits, max_seq_len 6;
#     one model per predictor, the three he ran: vpin, ofi_L10_norm_n,
#     micro_price.
#   group 1 — his LearningKraus_multivariate.py driver: the joint
#     (log_mid, ofi_L10_norm_n, micro_price, vpin) encoding, m = 4^4 = 256,
#     6 qubits, max_seq_len 4, adam, batch 6*512; 'L10_micro_vpin' is his
#     hand-written file-name tag.
MULTI_PREDICTOR = ["ofi_L10_norm_n", "micro_price", "vpin"]
TRAIN_GROUPS = [
    dict(suffix="", predictors=["vpin", "ofi_L10_norm_n", "micro_price"],
         n_qubits=3, batch_size=8*512, optimizer="sgd", max_seq_len=6,
         predictor_abbrev=""),
    dict(suffix="_multivariate", predictors=[MULTI_PREDICTOR],
         n_qubits=6, batch_size=6*512, optimizer="adam", max_seq_len=4,
         predictor_abbrev="L10_micro_vpin"),
]

# distribution stage: colleague's full spec (his email / cls_reference.py) —
# 10 bivariate predictors x 5 classes, v2 (-1,0,1) CLS order, plus the
# multivariate joint entry (SEQ only — he defines no multivariate CLS)
DIST_PREDICTORS = ["tvi_n", "obi_L1", "ofi_L1_n", "ofi_L1_n_norm",
                   "ofi_L1_norm_n", "ofi_L3_norm_n", "ofi_L10_norm_n",
                   "micro_price", "vpin", "sigma_W", MULTI_PREDICTOR]
# one CLS file per class per bivariate predictor. This is the dominant cost
# of the distribution stage: 10 predictors x 5 classes = 50 class-conditional
# passes per day, vs 11 sequence passes. Trim it to shorten stage 2.
CLS_NAMES = ["c1", "c2", "c4", "ca2", "ca4"]

EPOCHS = 3000        # both drivers: epochs=3000
SEED = None          # both drivers are unseeded; set an int for reproducible runs
# (lr=1e-3, nll_seq, learn_rho0=True, min_seq_prob=0.0, eval batch 2*1024 and
#  200 plotted entries come from the TrainingConfig defaults — identical in
#  both of his drivers. m is derived from the encoding: n_symbols^2 = 16
#  bivariate, n_symbols^4 = 256 multivariate. Everything is written into the
#  generated configs and printed below — nothing is hidden. device=auto picks
#  CUDA automatically. training.num_workers stays 0: his scripts pass 8, but
#  his train() hardcodes num_workers=0 in the DataLoader, so 0 is what his
#  runs actually did — the knob is results-neutral either way, and low values
#  are right while trainings are fanned across GPUs.)

# per-symbol raw-data directory: {} = use configs/default.yaml's
# data.asset_paths catalog as-is (NVDA/INTC -> data/NVDA_INTC, IBM -> data/IBM);
# override individual entries here, e.g. {"IBM": "/mnt/other/data/IBM"}
ASSET_PATH_OVERRIDES = {}
WORKERS = 0          # LINUX: one featurize worker per core  (Mac: use 4)
SKIP_DISTRIBUTIONS = False
RUN_ENSEMBLE = True  # LINUX: build the 20 v2 ENS_TD_* tables per symbol too

# stage 3 fan-out. The 3-qubit model is tiny (m=16 operators of d=8,
# ~2k params) and uses only a few % of an A100 (the 6-qubit multivariate
# model is bigger — m=256 operators of d=64 — but still nowhere near one
# full GPU), so training the 12 models one at a time leaves most of the
# box idle.
# Both of these are written into the generated config (as
# training.max_parallel / training.gpus) and read back from there by
# stage 3 -- they are the same fields `pipeline train-all` uses, so the
# schedule is reproducible from configs/april_*.yaml alone.
#   TRAIN_PARALLEL  0 = auto: one model per visible GPU (8 on the box)
#                   1 = sequential; N = exactly N at once (values above the
#                   GPU count are reasonable, each model barely occupies one)
#   GPUS            'auto' = every CUDA device torch sees (respects an
#                   externally set CUDA_VISIBLE_DEVICES); '0,2,5' = those
#                   ids only; 'none' = force CPU
TRAIN_PARALLEL = 0
GPUS = "auto"


In [ ]:
import json, sys, time, pathlib
ROOT = pathlib.Path.cwd()
for p in (ROOT, ROOT / "scripts", ROOT / "tests", ROOT / "TrainingDistributions"):
    sys.path.insert(0, str(p))

from run_april import ASSET_CATALOG, find_data_dir, april_dates, detect_pattern, make_config
from pipeline.config import RunConfig

ASSET_CATALOG.update(ASSET_PATH_OVERRIDES)

# each symbol resolves its own directory (NVDA/INTC share NVDA_INTC; IBM is
# separate); the day-glob only runs once per distinct directory
resolved = {s: find_data_dir(s) for s in SYMBOLS}
for s, d in resolved.items():
    assert d.is_dir(), f"data dir not found for {s}: {d}"

configs = {}        # "{symbol}{suffix}" -> path, one per (symbol, group)
dist_configs = {}   # symbol -> group-0 path (stage 2 runs once per symbol)
scope_by_dir = {}
for s, data_dir in resolved.items():
    if data_dir not in scope_by_dir:
        pattern = detect_pattern(data_dir)
        dates = april_dates(data_dir, pattern)
        scope_by_dir[data_dir] = (pattern, dates)
        print(f"data: {data_dir}  (pattern: {pattern})")
        print(f"April days: {len(dates)} ({dates[0]}..{dates[-1]})")
    pattern, dates = scope_by_dir[data_dir]
    for g in TRAIN_GROUPS:
        configs[s + g["suffix"]] = make_config(
            s, data_dir, dates, WORKERS, DIST_PREDICTORS, pattern,
            epochs=EPOCHS, n_qubits=g["n_qubits"],
            seed=-1 if SEED is None else SEED,
            train_predictors=g["predictors"],
            predicted=PREDICTED, cls_names=CLS_NAMES,
            gpus=GPUS, max_parallel=TRAIN_PARALLEL,
            batch_size=g["batch_size"], optimizer=g["optimizer"],
            max_seq_len=g["max_seq_len"],
            predictor_abbrev=g["predictor_abbrev"],
            config_suffix=g["suffix"])
    dist_configs[s] = configs[s]        # group 0 has suffix ""

# every parameter of the run, in full — nothing is implicit
for k, cfg_path in configs.items():
    print(f"\n{'='*30} {k}: {cfg_path.name} {'='*30}")
    print(cfg_path.read_text())

### Stage 2 — per-symbol distributions
One decode+featurize per (symbol, day), day-parallel; outputs land in
`outputs/april/{SYMBOL}/`. Skipped if `SKIP_DISTRIBUTIONS = True`.

In [ ]:
if not SKIP_DISTRIBUTIONS:
    from pipeline.runner import run
    # once per SYMBOL, not per config — the group configs share every
    # distribution setting, so one pass serves both training groups
    for symbol, cfg_path in dist_configs.items():
        cfg = RunConfig.load(cfg_path)          # banner derives from the CONFIG
        print(f"=== distributions: {symbol} ({len(cfg.data.dates)} days x "
              f"{len(cfg.distributions.predictors)} predictors x "
              f"{len(cfg.distributions.class_names) or 1} classes) ===")
        t0 = time.time()
        run(cfg, run_id=f"april-{symbol}")
        print(f"{symbol} done in {time.time()-t0:.0f}s -> outputs/april/{symbol}/\n")
else:
    print("skipped (SKIP_DISTRIBUTIONS=True)")

### Stage 2b (optional) — ensemble training tables (`ENS_TD_*`)
The colleague's fixed-length multi-channel experiment, **v2**
(`ensemble_training_data_2.py`): 3 bivariate channels + 1 joint multivariate
channel (log_mid + ofi_L10_norm_n + micro_price + vpin, 256-symbol alphabet),
timestamp-aligned, 5 sequence lengths × 4 class definitions → 20 pickles per
symbol under `outputs/april/{SYMBOL}/ensemble/` (`..._ALL` names). Counting
math is his code verbatim (byte-equivalence: `tests/verify_ensemble_v2.py`);
featurize is served from the same per-symbol cache as stage 2, so this adds
roughly 30 min per symbol warm. Enable with `RUN_ENSEMBLE = True`.

In [ ]:
if RUN_ENSEMBLE:
    from pipeline.ensemble import run_ensemble
    # once per SYMBOL — the group configs share every ensemble setting
    for symbol, cfg_path in dist_configs.items():
        cfg = RunConfig.load(cfg_path)          # banner derives from the CONFIG
        print(f"=== ensemble tables: {symbol} "
              f"({len(cfg.ensemble.predictors)} channels x "
              f"{len(cfg.ensemble.seq_lengths)} lengths x "
              f"{len(cfg.ensemble.class_names)} classes) ===")
        t0 = time.time()
        outputs = run_ensemble(cfg, run_id=f"april-ensemble-{symbol}")
        print(f"{symbol}: {len(outputs)} ENS_TD files in "
              f"{time.time()-t0:.0f}s -> outputs/april/{symbol}/ensemble/\n")
else:
    print("skipped (RUN_ENSEMBLE=False)")

### Stage 3 — the 12 models, fanned across GPUs (send-as-you-go)
Each model is trained by the **pipeline's registered trainer**
(`pipeline.models`, the same code path as `python -m pipeline train`), dispatched one
per GPU (`TRAIN_PARALLEL`). One job per (symbol, training group, predictor): 3
bivariate + 1 multivariate per symbol, each with its group's hyperparameters read
back from its own config. As each finishes, its **READY TO SEND** bundle is printed
and the 4 charts render inline — **in completion order, not job order**, so each block
names its symbol/predictor/device. Every model also gets its own run dir
(`outputs/runs/april-train-{key}-{predictor}/`) with `config.yaml` + `progress.json`,
watchable individually via `python -m pipeline status`. Aggregate results persist in
`outputs/april/april_summary.json`.

In [ ]:
from IPython.display import Image, display
from run_april import plan_training, run_training_jobs

summary_path = ROOT / "outputs" / "april" / "april_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
results = []

# schedule comes from the generated config (training.gpus /
# training.max_parallel), not from the notebook constants -- the printed
# YAML above is the authority for how stage 3 runs
jobs, n_par = plan_training(configs)        # one job per (symbol, predictor)
devices = sorted({j["device"] for j in jobs})

print(f"{len(jobs)} models | {len(devices)} device(s) | {n_par} at a time")
for j in jobs:                              # per-model params, not one symbol's
    t = RunConfig.load(j["config"]).training
    print(f"    {j['symbol']:6s} x {j['label']:24s} -> {j['device']}"
          f"  (epochs={t.epochs}, {t.n_qubits}q, batch={t.batch_size}, "
          f"lr={t.lr}, {t.optimizer}/{t.loss_kind}, "
          f"seed={'unseeded' if t.seed < 0 else t.seed})")

for i, r in enumerate(run_training_jobs(jobs, n_par), 1):
    results.append(r)
    summary_path.write_text(json.dumps(results, indent=1))

    ran_on = r["device"]
    if r.get("requested_device") not in (None, ran_on):
        ran_on = f"{ran_on} (requested {r['requested_device']})"
    print(f"\n>>> MODEL {i}/{len(jobs)} COMPLETE — "
          f"{r['symbol']} x {r['predictor']} on {ran_on} — READY TO SEND:")
    print(f"    result file 1: {r['model_file']}")
    print(f"    result file 2: {r['weights_file']}")
    for png in r["plots"]:
        print(f"    image:         {png}")
    if r.get("chart_error"):
        print(f"    NOTE: charts failed ({r['chart_error']}); "
              f"model + weights above are complete")
    print(f"    cost={r['loss']:.3e}  ({r['train_seconds']:.0f}s)")
    for png in r["plots"]:
        display(Image(filename=png))

print(f"\nAll {len(jobs)} models done. Summary: {summary_path}")

### Where everything lands
```
configs/april_{nvda,intc,ibm}.yaml                bivariate experiment definitions
configs/april_{...}_multivariate.yaml             multivariate experiment definitions
outputs/april/{SYMBOL}/SEQ_DISTR_*, CLS_*      per-symbol distributions (incl. the
                                               multivariate SEQ_DISTR_* file)
outputs/april/{SYMBOL}/feature_cache/          per-symbol featurized days
outputs/april/{SYMBOL}/ensemble/ENS_TD_*       20 v2 ensemble tables (stage 2b)
outputs/april/{SYMBOL}/models/MOD_*            model pickle   (1 of 2 per model)
outputs/april/{SYMBOL}/models/WGHTS_*.pt       weights        (2 of 2 per model)
outputs/april/{SYMBOL}/models/*_{n}q_1..4.png  4 charts per model
outputs/runs/april-train-{key}-{predictor}/    per-model config.yaml + progress.json
outputs/april/april_summary.json               cost + timing per model
```
Model files follow the colleague's current drivers verbatim:
`MOD_{sym}_bivariate_log_mid-{pred}_202504_3q` +
`WGHTS_{sym}_bivariate_log_mid-{pred}_202504_3q.pt`, and
`MOD_/WGHTS_{sym}_multivariate_log_mid-L10_micro_vpin_202504_6q(.pt)` — the
`WGHTS_` names carry **no `MOD_` infix**. Earlier conventions are retired:
pre-2026-07-20 batches used `MODR_*`/`WGHTS_MODR_*` (an off-by-one in the old
naming), and the 2026-07-20 consolidation briefly used `WGHTS_MOD_*` — filenames
from this run will match neither.